# EURUSDProphet: EUR/USD Exchange Rate Forecasting

This notebook contains analysis and mathematical models for predicting EUR/USD exchange rate movements using time series analysis and statistical methods.

## Contents
- Data loading and exploration
- Statistical analysis of EUR/USD trends
- Predictive modeling
- Visualization of forecasts

# EURUSDProphet: EUR/USD Exchange Rate Forecasting

This notebook contains analysis and mathematical models for predicting EUR/USD exchange rate movements using time series analysis and statistical methods.

## Contents
- Data loading and exploration
- Statistical analysis of EUR/USD trends
- Predictive modeling
- Visualization of forecasts

In [2]:
# 1. Data Loading and Descriptive Statistics
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from datetime import datetime, timedelta

# Set plot style
sns.set(style="whitegrid", palette="viridis")

# Define the ticker and time period
ticker = "EURUSD=X"
end_date = datetime.today()
start_date = end_date - timedelta(days=5*365)

# Download the data
data = yf.download(ticker, start=start_date, end=end_date)

print("Data downloaded successfully. Here are the first 5 rows:")
print(data.head())

[*********************100%***********************]  1 of 1 completed

Data downloaded successfully. Here are the first 5 rows:
Price          Close      High       Low      Open   Volume
Ticker      EURUSD=X  EURUSD=X  EURUSD=X  EURUSD=X EURUSD=X
Date                                                       
2021-03-16  1.192577  1.195300  1.188439  1.192734        0
2021-03-17  1.190165  1.191800  1.188764  1.190210        0
2021-03-18  1.198279  1.199100  1.191015  1.198064        0
2021-03-19  1.191824  1.193745  1.187493  1.191767        0
2021-03-22  1.188312  1.193745  1.187606  1.188453        0


## Daily Percentage Returns

The daily percentage return measures the discrete rate of change of the EUR/USD price. Mathematically, it can be seen as an approximation of the first derivative of the price function with respect to time, calculated over a discrete interval of one day. It is calculated as:

$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$

Where $P_t$ is the price at time $t$ and $P_{t-1}$ is the price at the previous time step. This value is fundamental for understanding the volatility and statistical properties of the asset.

In [ ]:
# 2. Calculate Daily Percentage Returns
data['Daily_Return'] = data['Close'].pct_change()

# Drop the first row since it will have a NaN value for the return
returns = data['Daily_Return'].dropna()

# 3. Calculate and Print Summary Statistics
mean_return = returns.mean()
median_return = returns.median()
variance_return = returns.var()
std_dev_return = returns.std(ddof=1) # Unbiased estimator
skewness = returns.skew()
kurt = returns.kurt() # This is excess kurtosis (Kurtosis - 3)

print("--- Summary Statistics for Daily Returns ---")
print(f"Mean: {mean_return:.6f}")
print(f"Median: {median_return:.6f}")
print(f"Variance: {variance_return:.6f}")
print(f"Standard Deviation: {std_dev_return:.6f}")
print(f"Skewness: {skewness:.6f}")
print(f"Kurtosis (Excess): {kurt:.6f}")
print("------------------------------------------")

In [ ]:
# 4. Plot Histogram of Returns vs. Normal Distribution
plt.figure(figsize=(12, 6))

# Plot the histogram of the returns
sns.histplot(returns, bins=100, kde=False, stat="density", label="Daily Returns Distribution", color='skyblue', alpha=0.6)

# Overlay a standard normal (Gaussian) distribution
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mean_return, std_dev_return)
plt.plot(x, p, 'k', linewidth=2, label='Normal Distribution')

# Add titles and labels
plt.title('Histogram of EUR/USD Daily Returns vs. Normal Distribution')
plt.xlabel('Daily Percentage Return')
plt.ylabel('Density')
plt.legend()

plt.show()

## 5. Interpretation of Skewness and Kurtosis

### Skewness
The calculated skewness for the daily returns is **{skewness:.4f}**.
- A value close to zero indicates a nearly symmetrical distribution.
- A negative value would indicate a left-skewed distribution (long tail on the left side), suggesting a higher probability of large negative returns than large positive ones.
- A positive value would indicate a right-skewed distribution (long tail on the right side).
Our result suggests a slight asymmetry in the returns distribution.

### Kurtosis and "Fat Tails"
The calculated excess kurtosis is **{kurt:.4f}**.
- A standard normal (Gaussian) distribution has a kurtosis of 3. "Excess kurtosis" is simply `Kurtosis - 3`.
- A positive excess kurtosis (greater than 0) indicates a **leptokurtic** distribution. This means the distribution has "fatter tails" and a sharper peak than a normal distribution.
- **"Fat tails"** imply that extreme market events (both large gains and large losses) are much more likely to occur than would be predicted by a perfect Gaussian model. The high kurtosis value observed here is a classic characteristic of financial returns data, highlighting the inherent risk of sudden, significant price movements that are underestimated by simpler models.
